# Attention on a real Transformer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/08-transformer-attention.ipynb)

**Reading.** [Tutorial 4](04-attention-and-alignment.ipynb) measured attention on a toy task with a known right answer, and got a clean anti-diagonal. This notebook runs the same inspection on a `SimpleTransformer` trained on real English-Spanish text, the kind of model you train for A8, reading a sentence it has never seen. The weights it shows are the decoder's cross-attention: where each target word looked in the source while the model wrote it.

By the end you will have:

1. translated a held-out sentence with a pretrained Transformer;
2. read its cross-attention as an alignment map, word by word;
3. seen a model find the right source words and still translate badly.

**Running in Google Colab?** Run the next two cells. The first installs TorchLingo; the second downloads the pretrained model, about 11 MB. There is nothing to uncomment, and no GPU is needed.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 8b (*Neural MT: The Transformer*). A head start on **A8**.

In [1]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo>=0.2.2"
import torchlingo

In [2]:
from torchlingo.colab import setup

# The pretrained model, its subword vocabulary and its held-out test set. A pip install does
# not ship them, so setup downloads them if they are missing.
env = setup(data=["data/pretrained/model.pt", "data/pretrained/spm.model", "data/pretrained/test.tsv"])

import torch

from torchlingo.config import get_default_config
from torchlingo.visualization import format_attention

CFG = get_default_config()

TorchLingo 0.2.3 | PyTorch 2.13.0 | device cpu | not in Colab
  data/pretrained/model.pt    10.2 MB
  data/pretrained/spm.model     0.3 MB
  data/pretrained/test.tsv     0.3 MB


## The model, and a sentence it has never seen

The checkpoint is the one [Tutorial 5](05-real-translations.ipynb) reads, and the sentence comes from its held-out test set. The cell translates it greedily, then recovers the attention for the translation the model **actually produced**, not for a reference fed back in: on a model whose output is wrong, those are different questions, and only the first shows what the model did.

PyTorch's Transformer layers never compute their attention weights by default, so `attention_for_sequence` recovers them. Rows are the model's output pieces, columns are the source pieces, and the shading runs `·` `░` `▒` `▓` `█`.

In [3]:
from pathlib import Path

import pandas as pd

from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import attention_for_sequence, greedy_decode
from torchlingo.models import SimpleTransformer

PRETRAINED = Path("data/pretrained")
SENTENCE = "This report is our main public report."

checkpoint = torch.load(PRETRAINED / "model.pt", map_location="cpu", weights_only=False)
sp_vocab = SentencePieceVocab(str(PRETRAINED / "spm.model"))
transformer = SimpleTransformer(
    src_vocab_size=checkpoint["vocab_size"],
    tgt_vocab_size=checkpoint["vocab_size"],
    **checkpoint["model_config"],
)
transformer.load_state_dict(checkpoint["model_state_dict"])
transformer.eval()

# Pinned by text rather than by position, so this cell shows the sentence the
# discussion below describes -- or fails loudly if the test set ever changes.
held_out = pd.read_csv(PRETRAINED / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
match = held_out[held_out.src == SENTENCE]
assert len(match) == 1, f"expected one row for {SENTENCE!r}, found {len(match)}"

source = torch.tensor([sp_vocab.encode(SENTENCE, add_special_tokens=True)])
produced = greedy_decode(transformer, source, max_len=30, config=CFG)[0]

# Attention for what the model produced, recovered with one teacher-forced pass
# over the finished sequence. Exact, not an approximation: the decoder is
# causally masked, so re-running reproduces each row as the decode computed it.
tfm_weights = attention_for_sequence(transformer, source, produced, config=CFG)

print("EN   ", SENTENCE)
print("MODEL", sp_vocab.decode(produced, skip_special_tokens=True))
print("REF  ", match.tgt.iloc[0])
print()
print(format_attention(
    tfm_weights,
    sp_vocab.indices_to_tokens(source[0].tolist()),
    # Drop <sos>: row n is the state that predicts token n + 1.
    sp_vocab.indices_to_tokens(list(produced))[1:],
))

EN    This report is our main public report.
MODEL Esta es nuestra pública de reporte.
REF   Este reporte es nuestro principal reporte público.

           <sos>   ▁This     ▁re    port     ▁is    ▁our     ▁ma      in ▁public     ▁re    port       .   <eos>
▁Esta        ···     ▓▓▓     ···     ···     ░░░     ···     ···     ···     ···     ···     ···     ···     ···
▁es          ···     ···     ···     ···     ░░░     ···     ···     ···     ░░░     ···     ···     ···     ···
▁nuestra     ···     ···     ···     ···     ···     ▒▒▒     ···     ···     ···     ···     ···     ···     ···
▁públ        ···     ···     ···     ···     ···     ···     ···     ···     ▒▒▒     ···     ···     ···     ···
ica          ···     ···     ░░░     ···     ···     ···     ░░░     ···     ···     ░░░     ···     ···     ···
▁de          ···     ···     ···     ···     ···     ░░░     ···     ···     ░░░     ···     ···     ░░░     ···
▁re          ···     ···     ···     ░░░     ···     ░░░     ···

## Read it against the translation

The output is wrong: `Esta es nuestra pública de reporte.` against a reference of `Este reporte es nuestro principal reporte público.` But look at where the attention actually went, because the obvious conclusion is not the right one.

**Attention found the right words.** `▁Esta` peaks on `▁This`. `▁nuestra` peaks on `▁our`. `▁públ` peaks on `▁public`. `port` peaks on the `port` piece of "report". Four content words, four correct alignments, learned from nothing but examples, on a sentence the model has never seen, with a vocabulary of subword pieces rather than words.

**So the failure is not in attention.** It is in what the decoder does with what attention handed it. The model retrieves `public`, `our` and `report` correctly and then assembles them into `nuestra pública de reporte`: Spanish words in an order Spanish does not use, with `de` appearing from nowhere. Knowing *where to look* and knowing *what to write* are separate abilities, and this model has more of the first than the second.

**Look at the word that disappeared.** "main" is split into the pieces `▁ma` and `in`, and those two columns are the faintest in the grid: no row ever peaks on either. Sure enough, "main" has no counterpart in the output. A dropped word often shows up here first, as a column nothing looks at.

**Compare the shapes.** Tutorial 4's map is a clean anti-diagonal, one `███` per row, because its task has exactly one right answer per position. This map has a handful of `▓▓▓` and `▒▒▒` peaks in a wash of `░░░`. That diffuse texture is what attention on real data normally looks like; an anti-diagonal is the exception, and only because that task was built to have one.

## What's next

- **Before you plot attention, call `model.eval()`.** In training mode dropout zeroes attention weights, the rows stop summing to 1, and the plot still looks plausible.
- **On your own A8 model:** the two calls that matter, `attention_for_sequence` and `format_attention`, take any trained `SimpleTransformer` with its own source and target vocabularies. `greedy_decode` and `beam_search_decode` also take `return_attention=True`, which returns this map from the decode itself.
- **Every layer and head:** `attention_for_sequence` averages the heads of the last decoder layer; [`capture_cross_attention`](../reference/visualization.md) gives all of them. For the numbers rather than the shading, `format_attention(..., show_values=True)`.